# 09. So sánh kết quả 4 mô hình

Notebook đọc kết quả đánh giá trên cùng tập Test WELFake từ Notebook 04, 06, 07 và 08. Quy ước nhãn thống nhất: `0 = Tin giả`, `1 = Tin thật`. Các chỉ số Precision, Recall và F1-score lấy lớp dương `1` (Tin thật).

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
RESULT_DIR = "/content/drive/MyDrive/fake-news-detection/results"

metric_files = {
    "Naive Bayes": "naive_bayes_metrics.json",
    "CNN": "cnn_metrics.json",
    "RNN-LSTM": "rnn_metrics.json",
    "BERT": "bert_metrics.json"
}

missing_files = [name for name in metric_files.values()
                 if not os.path.exists(os.path.join(RESULT_DIR, name))]
if missing_files:
    raise FileNotFoundError(
        "Thiếu kết quả của một hoặc nhiều mô hình: " + ", ".join(missing_files) +
        ". Hãy chạy lại Notebook 04, 06, 07, 08 để tạo đủ metrics trước khi so sánh."
    )

all_results = []
required_metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
for model_name, filename in metric_files.items():
    path = os.path.join(RESULT_DIR, filename)
    with open(path, "r", encoding="utf-8") as f:
        item = json.load(f)
    missing_keys = [key for key in required_metrics if key not in item]
    if missing_keys:
        raise ValueError(f"{filename} thiếu các trường: {missing_keys}")
    item["Model"] = model_name
    all_results.append(item)

print("Đã đọc đủ kết quả của 4 mô hình:", [x["Model"] for x in all_results])

In [ ]:
df = pd.DataFrame(all_results)
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
comparison = df.set_index("Model").loc[
    ["Naive Bayes", "CNN", "RNN-LSTM", "BERT"], metrics
]

print("BẢNG SO SÁNH 4 MÔ HÌNH TRÊN CÙNG TẬP TEST WELFake")
display(comparison.round(4))

In [ ]:
comparison.plot(kind="bar", figsize=(12, 6), ylim=(0, 1), rot=0)
plt.title("So sánh hiệu suất 4 mô hình trên WELFake")
plt.xlabel("Mô hình")
plt.ylabel("Giá trị đánh giá")
plt.legend(loc="lower right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
best_model = df.loc[df["F1-score"].idxmax()]
print("MÔ HÌNH CÓ F1-SCORE CAO NHẤT")
print("Tên mô hình:", best_model["Model"])
for metric in metrics:
    print(f"{metric}: {best_model[metric]:.4f}")

In [ ]:
output_path = os.path.join(RESULT_DIR, "model_comparison.csv")
comparison.to_csv(output_path, encoding="utf-8-sig")
print("Đã lưu bảng so sánh:", output_path)